<a href="https://colab.research.google.com/github/zach-yan/ORFE-Network-Project/blob/main/ORF_387_Final_Project_Merge_Nodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Map the labels back to the original ones and combine the nodes csv and the labels csv.

In [ ]:
import pandas as pd
import ast

# 1. Load the data
profiles_df = pd.read_csv("author_classified_profiles.csv")

# Load the existing nodes
nodes_df = pd.read_csv("orfe_nodes_fuzzy_cleaned.csv")

# 2. Convert string representations of lists back into actual Python lists
profiles_df['Top_Subjects'] = profiles_df['Top_Subjects'].apply(ast.literal_eval)
profiles_df['Top_Applications'] = profiles_df['Top_Applications'].apply(ast.literal_eval)

# 3. Define the Reverse Mappings
subject_map = {
    "Mathematical Optimization and Convex Programming": "Optimization",
    "Statistics and Applied Machine Learning": "Statistics and Machine Learning (ML)",
    "Applied Probability and Stochastic Systems": "Probability",
    "Game Theory and Mechanism Design": "Game Theory",
    "Financial Engineering and Quantitative Finance": "Financial Engineering"
}

application_map = {
    "Pure Mathematical Theory": "Pure Theory",
    "Financial Markets and Asset Pricing": "Finance",
    "Healthcare Operations and Clinical Decision Support": "Healthcare",
    "Natural Language Processing and Text Analytics": "Natural Language Processing",
    "Network and Telecommunications Operations": "Networks/Telecommunications",
    "Transportation Networks and Logistics Planning": "Transportation/Logistics",
    "Supply Chain Management and Revenue Optimization": "Supply Chain/Revenue Management",
    "Robotics and Autonomous Systems": "Robotics",
    "Energy Systems and Power Grids": "Energy",
    "Public Policy and Social Operations": "Public Policy"
}

# 4. Apply the mappings
def condense_labels(label_list, mapping_dict):
    """Maps a list of expanded descriptions to their condensed versions."""
    return [mapping_dict.get(label, label) for label in label_list]

profiles_df['Subjects'] = profiles_df['Top_Subjects'].apply(lambda x: condense_labels(x, subject_map))
profiles_df['Applications'] = profiles_df['Top_Applications'].apply(lambda x: condense_labels(x, application_map))

# 5. Merge the classifications back into the main network nodes
# The Semantic Scholar ID is 'Id' in nodes_df and 'Author_Id' in profiles_df
final_nodes_df = pd.merge(
    nodes_df,
    profiles_df[['Author_Id', 'Subjects', 'Applications']],
    left_on='Id',
    right_on='Author_Id',
    how='left'  # Use a left join to keep all nodes, even if they lacked papers for inference
)

# Clean up the redundant merge column and fill empty lists for authors who had no papers
final_nodes_df.drop(columns=['Author_Id'], inplace=True)
final_nodes_df['Subjects'] = final_nodes_df['Subjects'].apply(lambda x: x if isinstance(x, list) else [])
final_nodes_df['Applications'] = final_nodes_df['Applications'].apply(lambda x: x if isinstance(x, list) else [])

# 6. Save the final enriched network!
final_nodes_df.to_csv("orfe_nodes_with_classifications.csv", index=False)

# Preview the results
print(final_nodes_df.head())

           Id                      Label  Layer Affiliation  \
0  1414099319  &#x00C2;ndrei Camponogara    2.0     Unknown   
1  1422426196               '. Gonz'alez    2.0     Unknown   
2  2226784417   'Alex Hern'andez-Garc'ia    2.0     Unknown   
3  2130393980                      A. A.    2.0     Unknown   
4    40804344               A. A. Abello    2.0     Unknown   

                      Subjects                                Applications  
0  [Optimization, Probability]  [Pure Theory, Networks/Telecommunications]  
1   [Probability, Game Theory]                [Pure Theory, Public Policy]  
2                           []                                          []  
3                           []                                          []  
4                           []                                          []  
